### Process Address Data
1. Ingest the data into the data lakehouse - bronze_addresses
2. Perform data quality checks and transform the data as required - silver_addresses_clean
3. Apply changes to the Addresses data (SCD Type 2) - silver_addresses

![image_1780979519849.png](./image_1780979519849.png "image_1780979519849.png")

### 1. Ingest the data into the data lakehouse - bronze_addresses

In [0]:
!pip install dlt

In [0]:
import dlt
from pyspark.sql.functions import *

In [0]:
@dlt.table(
    name='bronze_addresses',
    table_properties={'quality' :'bronze'},
    comment='Raw Addressses data ingested from the source system'
)
def create_bronze_addresses():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.inferColumnTypes","true")
            .load("/Volumes/catalog_circuitbox/landing/operational_data/addresses/")
            .select(
                "*",
                col("_metadata.file_path").alias("input_file_path"),
                current_timestamp().alias("ingest_timestamp")
            )
    )

### 2.Perform data quality checks and transform the data as required - silver_addresses_clean

![image_1780985930339.png](./image_1780985930339.png "image_1780985930339.png")

In [0]:
@dlt.table(
    name='silver_address_clean',
    comment='Cleaned address data',
    table_properties={'quality':'silver'}
)
@dlt.expect_or_fail("valid_customer_id","customer_id IS NOT NULL")
@dlt.expect_or_drop("valid_address", "address_line_1 IS NOT NULL")
@dlt.expect("valid_postcode","LENGTH(postcode)=5")
def create_silver_addresses_clean():
    return(
        spark.readStream.table("LIVE.bronze_addresses")
            .select(
                "customer_id",
                "address_line_1",
                "city",
                "state",
                "postcode",
                col("created_date").cast("date")
            )
    )

### 3.Apply changes to the Addresses data (SCD Type 2) - silver_addresses

![image_1780995828410.png](./image_1780995828410.png "image_1780995828410.png")

In [0]:
dlt.create_streaming_table(
    name='silver_addresses',
    comment='SCD Type 2 addresses data',
    table_properties={'quality' : 'silver'}
)

In [0]:
dlt.apply_changes(
    target='silver_addresses',
    source='silver_address_clean',
    keys=['customer_id'],
    sequence_by='created_date',
    stored_as_scd_type=2
)